In [ ]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_repo_root = next(path for path in (_here, *_here.parents) if (path / "REMAKE" / "configs" / "cfg.py").exists())
os.chdir(_repo_root)
sys.path.insert(0, str(_repo_root))

import matplotlib.pyplot as plt
import pandas as pd
from pprint import pprint
from IPython.display import Markdown, display

from REMAKE.data.share_data import load_share_data
from REMAKE.scripts.madrl import SCHEMES, plot_madrl_outputs, run_madrl_experiments
from REMAKE.utils.run_artifacts import load_experiment_context


In [ ]:
run_dir = Path("artifacts/runs/20260504_193738_6ac06409")
train_episodes = 500

cfg, run_dir = load_experiment_context(run_dir)
share_data = load_share_data(run_dir / "share_data", cfg)

scheme_rows = []
for item in SCHEMES:
    w_voltage, w_line, w_trafo = item["reward"]
    scheme_rows.append({
        "scheme": item["scheme"],
        "controller": item["controller"],
        "algo": item["algo"],
        "projection": item["projection"],
        "train_episodes": int(train_episodes),
        "action_dim": int(cfg.model.action_dim),
        "max_charge_rate": float(cfg.env.max_charge_rate),
        "init_soc": float(cfg.env.init_soc),
        "train_init_soc_low": float(cfg.env.train_init_soc_low),
        "train_init_soc_high": float(cfg.env.train_init_soc_high),
        "batch_size": int(cfg.train.batch_size),
        "gamma": float(cfg.algo.gamma),
        "tau": float(cfg.algo.tau),
        "hidden_dim": int(cfg.model.hidden_dim),
        "n_step_return": int(cfg.train.n_step_return),
        "w_voltage_pen": float(w_voltage),
        "w_line_pen": float(w_line),
        "w_trafo_pen": float(w_trafo),
    })

display(Markdown("# REMAKE MADRL"))
display(pd.DataFrame([{
    "run_dir": str(run_dir),
    "device": cfg.runtime.device,
    "eval_start": cfg.data.eval_start_date,
    "eval_end": cfg.data.eval_end_date,
    "eval_episodes": int(share_data.eval["price"].shape[0]),
    "episode_steps": int(cfg.env.episode_steps),
    "sequence_length": int(cfg.obs.sequence_length),
}]))
display(pd.DataFrame(scheme_rows))


In [ ]:
result = run_madrl_experiments(cfg, run_dir, share_data, episodes=train_episodes)

display(Markdown("## Train Summary"))
display(result["train_summary"])

for scheme, saved in result["records"].items():
    display(Markdown(f"## {scheme}"))
    display(saved["metrics_df"])
    display(saved["rollout"].summary)
    pprint({"record_dir": str(saved["record_dir"])})


In [ ]:
plt.close("all")
figures = plot_madrl_outputs(result, run_dir)
_plot_specs = [
    ("1. 电力电量平衡", "madrl_power_balance"),
    ("2. 电价", "madrl_price"),
    ("3. 总体储能充放功率与 SoC", "madrl_battery_soc"),
    ("4. 电压", "madrl_voltage"),
    ("5. 净负荷", "madrl_net_load"),
    ("6. 训练奖励", "madrl_learning_curve"),
]
for title, name in _plot_specs:
    print(title)
    display(figures[name])
    plt.close(figures[name])
